In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import requests

In [12]:
#insta
options = Options()                # Create Chrome browser configuration
options.add_argument("--headless")   #runs Chrome w/o opening a visible window
# prevent being detecting as bot
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/547.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
driver = webdriver.Chrome(options=options)   #Chrome browser control by Selenium
try:
    url = "https://instrack.app/top-accounts"
    driver.get(url)

    # loading page
    wait = WebDriverWait(driver,15)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
    
    soup = BeautifulSoup(driver.page_source,'html.parser')    # Get HTML on loaded page and parses the HTML to extract elements
    table1 = soup.find('table')
    
    if table1:
        data = []
        rows = table1.find_all('tr')    # Collect all rows from the table (tr = table rows)
        #Extract each column
        for row in rows:
            cols = [td.get_text(strip=True) for td in row.find_all(['td','th'])]         # td= table cells, th = header, get_text(strip=True) remove extra spaces
            if len(cols) >= 4: #filter valid row
                data.append(cols)
   
        # Convert data into a DataFrame
        df_instrack = pd.DataFrame(data[1:], columns=["Rank", "Account", "Followers", "Following", "Posts", "Engagement"])
        df_instrack = df_instrack[["Account", "Followers", "Following", "Posts"]]
        print("Success! Data saved.")
    else:
        print("Table still not found.")

# closing browser
finally:
    driver.quit()

df_instrack

Success! Data saved.


,Account,Followers,Following,Posts
0,Instagram,"700,637,774",219,"8,366"
1,Cristiano Ronaldo,"672,205,704",629,"4,021"
2,Leo Messi,"511,784,846",357,"1,464"
3,Selena Gomez,"415,268,254",362,"2,158"
4,Kylie,"390,854,505",132,"7,376"
...,...,...,...,...
95,Vinicius Jr. ⚡️🇧🇷,"59,175,402","1,176",744
96,BLΛƆKPIИK,"58,927,040",5,"2,097"
97,Chris Hemsworth,"58,864,262",757,"1,288"
98,Kriti Sanon 🦋,"58,670,704",422,"2,756"


In [24]:
# ViralPitch
# Configure selenium browser
options = Options()
options.add_argument("--headless")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/547.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
driver = webdriver.Chrome(options=options)
try:
    #Open ViralPitch link
    url = "https://viralpitch.co/topinfluencers/top-100-instagram-influencers-in-us/"
    driver.get(url)

    # Loading element
    wait = WebDriverWait(driver,15)
    wait.until(EC.presence_of_element_located((By.ID, "influencerTableBody")))

    #Parse HTML with BeautifulSoup
    soup = BeautifulSoup(driver.page_source,'html.parser')    
    table2 = soup.find('tbody', id='influencerTableBody')
    
    if table2:
        data = []
        #Extract rows
        rows = table2.find_all('tr')
        #Extract username and name
        for row in rows:
            cells = row.find_all('td')
            if len(cells) >= 7: 
                user = row.find('a', class_='user-id')
                username = user.get_text(strip=True) if user else "N/A"

                name = row.find('a', class_='user-name')
                full_name = name.get_text(strip=True) if name else "N/A"

                # Save data
                data.append([username, full_name])
        # Convert DataFrame
        df_viralpitch = pd.DataFrame(data, columns=["Username", "Name"])
        print("Success! Data saved.")
    else:
        print("Table still not found.")

# Close browser
finally:
    driver.quit()

df_viralpitch

Success! Data saved.


,Username,Name
0,instagram,Instagram
1,cristiano,Cristiano Ronaldo
2,leomessi,Leo Messi
3,kyliejenner,Kylie
4,selenagomez,Selena Gomez
...,...,...
95,mosalah,Mohamed Salah
96,tatawerneck,Tata Werneck
97,sunnyleone,Sunny Leone
98,sergioramos,Sergio Ramos


In [25]:
# Hopper
# Configure selenium browser
options = Options()
options.add_argument("--headless")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/547.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
driver = webdriver.Chrome(options=options)
try:
    #Open Hopper link
    url_hopper = "https://www.hopperhq.com/instagram-rich-list/"
    driver.get(url_hopper)

    # Loading element
    wait = WebDriverWait(driver, 15)
    wait.until(EC.presence_of_element_located((By.CLASS_NAME, "w-dyn-items")))

    #Parse HTML with BeautifulSoup
    soup_hopper = BeautifulSoup(driver.page_source, 'html.parser')
    items = soup_hopper.find_all('div', class_='w-dyn-item')

    # Extract name & username
    hopper_list = []
    for item in items:
        name_div = item.find('div', attrs={'bind':"317164bb-84aa-626e-0368-2cab86e55bd8"})
        user_div = item.find('div', class_='at_username')

        # Store results
        if name_div and user_div:
            hopper_list.append({
                "name": name_div.get_text(strip=True),
                "username": user_div.get_text(strip=True)
            })
    df_hopper = pd.DataFrame(hopper_list)
except Exception as e:
    print(f"Error scraping Hopper HQ: {e}")

df_hopper

,name,username
0,Cristiano Ronaldo,cristiano
1,Lionel Messi,leomessi
2,Selena Gomez,selenagomez
3,Kylie Jenner,kyliejenner
4,Dwayne Johnson,therock
...,...,...
95,Dan Bilzerian,danbilzerian
96,Johnny Depp,johnnydepp
97,Sergio Aguero,kunaguero
98,Floyd Mayweather,floydmayweather


In [26]:
# Merging Datasets

# Clean text for matching
df_instrack['Account_Clean'] = df_instrack['Account'].str.strip().str.lower() 
df_viralpitch['Name_Clean'] = df_viralpitch['Name'].str.strip().str.lower() 
df_hopper['Name_Clean'] = df_hopper['name'].str.strip().str.lower()

# Merge Instrack & ViralPitch
df = pd.merge( 
    df_instrack, 
    df_viralpitch[['Name_Clean', 'Username']], 
    left_on='Account_Clean', 
    right_on='Name_Clean', 
    how='left' 
)

# Merge w/ Hopper
df_final = pd.merge( 
    df, 
    df_hopper[['Name_Clean', 'username']], 
    left_on='Account_Clean', 
    right_on='Name_Clean', 
    how='left', 
    suffixes=('', '_drop') 
)

# Fill missing username
df_final['Username'] = df_final['Username'].fillna(df_final['username'])
# Removing unecessary columns
df = df_final.drop(columns=['Account_Clean', 'Name_Clean', 'username', 'Name_Clean_drop'], errors='ignore') 
df

,Account,Followers,Following,Posts,Username
0,Instagram,"700,637,774",219,"8,366",instagram
1,Cristiano Ronaldo,"672,205,704",629,"4,021",cristiano
2,Leo Messi,"511,784,846",357,"1,464",leomessi
3,Selena Gomez,"415,268,254",362,"2,158",selenagomez
4,Kylie,"390,854,505",132,"7,376",kyliejenner
...,...,...,...,...,...
97,Vinicius Jr. ⚡️🇧🇷,"59,175,402","1,176",744,NaN
98,BLΛƆKPIИK,"58,927,040",5,"2,097",NaN
99,Chris Hemsworth,"58,864,262",757,"1,288",chrishemsworth
100,Kriti Sanon 🦋,"58,670,704",422,"2,756",NaN


In [42]:
df.to_csv("insta_influencers.csv", index=False)